# 01 — Eksploracja danych Yale EMMLC

Cel: poznać dataset, rozkłady kluczowych zmiennych, wzorce braków danych.

Wymagania: uruchomione `scripts/01_convert_rdata.py` (parquet w `data/processed/yale_emmlc_raw.parquet`).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.load_data import load_dataset
from src.data.preprocessing import build_feature_groups
from src.data.esi_mts_mapping import map_dataframe_to_mts

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

In [ ]:
df = load_dataset()
print(f'Shape: {df.shape}')
df.head()

## Rozkład ESI

In [ ]:
if 'esi' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    df['esi'].value_counts().sort_index().plot.bar(ax=axes[0], color='steelblue')
    axes[0].set_title('Rozkład ESI (liczność)')
    axes[0].set_xlabel('ESI')
    df['esi'].value_counts(normalize=True).sort_index().plot.bar(ax=axes[1], color='coral')
    axes[1].set_title('Rozkład ESI (procent)')
    plt.tight_layout()
    plt.show()

## Mapowanie ESI → MTS

In [ ]:
df_mts = map_dataframe_to_mts(df.copy(), use_enhanced=True)
df_mts['mts_color'].value_counts()

## Grupy cech

In [ ]:
groups = build_feature_groups(df_mts)
summary_df = pd.DataFrame.from_dict(groups.summary(), orient='index', columns=['count'])
summary_df

## Vital signs — rozkłady

In [ ]:
vitals = [c for c in groups.triage_vitals if c in df_mts.columns]
if vitals:
    fig, axes = plt.subplots(nrows=(len(vitals) + 1) // 2, ncols=2, figsize=(14, 3 * ((len(vitals) + 1) // 2)))
    for ax, vital in zip(axes.flatten(), vitals):
        df_mts[vital].dropna().hist(ax=ax, bins=50, color='steelblue', edgecolor='navy')
        ax.set_title(vital)
    plt.tight_layout()
    plt.show()

## Procent braków danych — top 30 kolumn

In [ ]:
missing_pct = (df_mts.isna().sum() / len(df_mts)).sort_values(ascending=False).head(30)
fig, ax = plt.subplots(figsize=(10, 8))
missing_pct.plot.barh(ax=ax, color='salmon')
ax.set_title('Top 30 kolumn z największym % braków')
ax.invert_yaxis()
plt.tight_layout()
plt.show()